# Notebook 6: common-factor diagnostics

Principal-component and selected-design collinearity diagnostics.

In [1]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *

import numpy as np
import pandas as pd

selection = load_result("nb0_selection")["selection"]
P = int(selection["p"])
K = int(selection["k"])

d = load_data(network="geographic")
Y_train = d["Y_train"].to_numpy()
N = d["N"]
W = knn_sparsify(d["networks"]["geographic"], K)
STAGES = [2] * P

## Principal components

In [2]:
Yc = Y_train - Y_train.mean(axis=0, keepdims=True)
U, S, Vt = np.linalg.svd(Yc, full_matrices=False)
var_share = S**2 / np.sum(S**2)
if Vt[0][np.argmax(np.abs(Vt[0]))] < 0:
    Vt[0] = -Vt[0]
    U[:, 0] = -U[:, 0]

factor = U[:, 0] * S[0]
loadings = Vt[0]
mean_series = Yc.mean(axis=1)
corr_pc1_mean = float(np.corrcoef(factor, mean_series)[0, 1])
loading_cv = float(np.std(loadings) / np.abs(np.mean(loadings)))
loading_ratio = float(np.max(np.abs(loadings)) / np.min(np.abs(loadings)))

Ys = Yc / Yc.std(axis=0, keepdims=True)
Us, Ss, Vts = np.linalg.svd(Ys, full_matrices=False)
var_share_std = Ss**2 / np.sum(Ss**2)
if Vts[0][np.argmax(np.abs(Vts[0]))] < 0:
    Vts[0] = -Vts[0]
    Us[:, 0] = -Us[:, 0]

loadings_std = Vts[0]
corr_pc1_mean_std = float(np.corrcoef(Us[:, 0] * Ss[0], Ys.mean(axis=1))[0, 1])
loading_cv_std = float(np.std(loadings_std) / np.abs(np.mean(loadings_std)))
loading_ratio_std = float(np.max(np.abs(loadings_std)) / np.min(np.abs(loadings_std)))

print(f"PC1 variance share={var_share[0]:.5f}")
print(f"PC1-PC3 cumulative share={var_share[:3].sum():.5f}")
print(f"corr(PC1, cross-sectional mean)={corr_pc1_mean:.5f}")
print(f"PC1 loading CV={loading_cv:.5f}, max/min absolute loading={loading_ratio:.5f}")
print(f"standardised PC1 share={var_share_std[0]:.5f}")
print(f"standardised loading CV={loading_cv_std:.5f}, max/min={loading_ratio_std:.5f}")

PC1 variance share=0.88829
PC1-PC3 cumulative share=0.95378
corr(PC1, cross-sectional mean)=0.96354
PC1 loading CV=1.57773, max/min absolute loading=88.36721
standardised PC1 share=0.57955
standardised loading CV=0.18860, max/min=2.91920


## Selected-design VIF

In [3]:
X, _, names = build_design(Y_train, W, p=P, stages=STAGES, mode="global_gnar", h=1)
i_own = names.index("own_lag1")
i_s1 = names.index("net_lag1_stage1")
i_s2 = names.index("net_lag1_stage2")

row_country = np.tile(np.arange(N), X.shape[0] // N)
def per_country_corr(a, b):
    vals = []
    for i in range(N):
        mask = row_country == i
        vals.append(np.corrcoef(X[mask, a], X[mask, b])[0, 1])
    return np.asarray(vals)

vif_s1, r2_s1 = variance_inflation_factor(X, i_s1)
vif_s2, r2_s2 = variance_inflation_factor(X, i_s2)
corr_s1_own = per_country_corr(i_s1, i_own)
corr_s2_own = per_country_corr(i_s2, i_own)
corr_s1_s2 = float(np.corrcoef(X[:, i_s1], X[:, i_s2])[0, 1])

print(f"stage-1 first-lag VIF={vif_s1:.5f}, R2={r2_s1:.5f}")
print(f"stage-2 first-lag VIF={vif_s2:.5f}, R2={r2_s2:.5f}")
print(f"corr(stage1, stage2)={corr_s1_s2:+.5f}")

stage-1 first-lag VIF=104.36010, R2=0.99042
stage-2 first-lag VIF=71.71091, R2=0.98606
corr(stage1, stage2)=+0.37556


/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/.RPvenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [4]:
save_result("nb6_common_factor", {
    "pc1_share": float(var_share[0]),
    "pc1_3_share": float(var_share[:3].sum()),
    "pc1_loadings": loadings.tolist(),
    "pc1_score": factor.tolist(),
    "corr_pc1_cross_sectional_mean": corr_pc1_mean,
    "pc1_loading_cv": loading_cv,
    "pc1_loading_max_min_ratio": loading_ratio,
    "pc1_share_standardised": float(var_share_std[0]),
    "pc1_3_share_standardised": float(var_share_std[:3].sum()),
    "pc1_loadings_standardised": loadings_std.tolist(),
    "corr_pc1_cross_sectional_mean_standardised": corr_pc1_mean_std,
    "pc1_loading_cv_standardised": loading_cv_std,
    "pc1_loading_max_min_ratio_standardised": loading_ratio_std,
    "selected_design": {
        "stage1_vif": vif_s1, "stage1_r2": r2_s1,
        "stage2_vif": vif_s2, "stage2_r2": r2_s2,
        "stage1_stage2_corr": corr_s1_s2,
        "stage1_own_corr_mean": float(corr_s1_own.mean()),
        "stage1_own_corr_min": float(corr_s1_own.min()),
        "stage1_own_corr_max": float(corr_s1_own.max()),
        "stage2_own_corr_mean": float(corr_s2_own.mean()),
        "stage2_own_corr_min": float(corr_s2_own.min()),
        "stage2_own_corr_max": float(corr_s2_own.max()),
    },
    "config": run_config(p=P, stages=STAGES, network="geographic", k=K,
                         max_stage=2, purpose="common_factor_diagnostics"),
})
print("saved nb6_common_factor")

saved nb6_common_factor
